# 06 — Conversational Interface

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Turn plain-language questions into the right analysis via intent detection and routing.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### Intent detection

In [2]:
from src.conversation.intent_classifier import IntentClassifier
ic = IntentClassifier()
for q in ["what is the sentiment?","who is mentioned?","summarize this","what topics are there","find similar articles about elections"]:
    print(f"{q:45s} -> {ic.classify(q)}")

what is the sentiment?                        -> sentiment
who is mentioned?                             -> entities
summarize this                                -> summarize
what topics are there                         -> topics
find similar articles about elections         -> search


### Route a question to the right module

In [3]:
from src.conversation.query_processor import QueryProcessor
from src.newsbot import NewsBot
from src.language_models.summarizer import Summarizer
from src.language_models.embeddings import SemanticSearch
from src.analysis.topic_modeler import TopicModeler

bot = NewsBot().train(df["text"], df["category"])
search = SemanticSearch().index(df["text"])
tm = TopicModeler(n_topics=5); tm.fit_transform(df["text"])
qp = QueryProcessor(bot=bot, summarizer=Summarizer(), search=search, topic_modeler=tm)

article = "The United Kingdom faces political uncertainty after the prime minister resigned."
for q in ["what is the sentiment?","who is mentioned?","summarize this"]:
    print(q, "->", qp.process(q, article))

what is the sentiment? -> {'intent': 'sentiment', 'response': 'The overall tone is negative (compound score -0.153, emotion: neutral).', 'raw': {'compound': -0.153, 'label': 'negative', 'polarity': 0.0, 'subjectivity': 0.1, 'emotion': 'neutral'}}
who is mentioned? -> {'intent': 'entities', 'response': 'Entities detected: United Kingdom (GPE).', 'raw': [('United Kingdom', 'GPE')]}
summarize this -> {'intent': 'summarize', 'response': 'Summary: The United Kingdom faces political uncertainty after the prime minister resigned.', 'raw': 'The United Kingdom faces political uncertainty after the prime minister resigned.'}


**Takeaway.** Because every capability is a clean class, a small intent classifier plus a router is enough to answer natural-language questions. This is the layer that ties the system together.